In [1]:
import torch
import os
from torch import nn
from monai.data import DataLoader, Dataset, CacheDataset, decollate_batch
from monai.losses import DiceCELoss, HausdorffDTLoss, TverskyLoss
from lightning.pytorch.callbacks import LearningRateMonitor, ModelCheckpoint
from lightning.pytorch.loggers import TensorBoardLogger
from monai.networks.nets import UNet
import lightning as L
from torch.optim import Adam
from torch.utils.tensorboard import SummaryWriter
from torch.optim.lr_scheduler import ReduceLROnPlateau
from glob import glob
import matplotlib.pyplot as plt
from monai.metrics import DiceMetric
from monai.visualize import plot_2d_or_3d_image
from monai.visualize.utils import blend_images

In [2]:
torch.cuda.is_available()

True

In [3]:
torch.__version__

'2.8.0+cu126'

In [4]:
import monai.transforms as t

In [5]:
dataset_path = r"F:\Datasets\Cell Contraction\segmentation_dataset"
validation_series = ["Cd_6s_2021", "Chymo_6s_2024", "CytoD_6s_2021-2"]

In [6]:
def load_data_lists(dataset_path, validation_series):
    available_series = os.listdir(os.path.join(dataset_path, "masks"))
    print(f"Available seried: {available_series}")

    train_data = []
    val_data = []
    for series in available_series:
        series_samples = []
        for mask_path in glob(os.path.join(dataset_path, "masks", series, "*.png")):            
            file_name = os.path.splitext(os.path.basename(mask_path))[0]
            image_path = os.path.join(dataset_path, series, f"{file_name}.JPG")
            if not os.path.exists(image_path):
                print(f"Warning: cannot find image for mask {series}:{file_name}")
                continue
            sample = {
                "image": image_path,
                "label": mask_path,
                "file_name": file_name,
                "series_name": series
            }
            series_samples.append(sample)
            
        if series in validation_series:
            val_data.extend(series_samples)
        else:
            train_data.extend(series_samples)

    return train_data, val_data


train_dataset_list, val_dataset_list = load_data_lists(dataset_path, validation_series)

Available seried: ['1A116_6s_2024', 'Ca-Mg_6s_2024', 'Cd_6s_2021', 'Chymo_3s_2021-1', 'Chymo_6s_2024', 'Chymotrypsin', 'CMFSS_6s_2021', 'CytoD_6s_2021-1', 'CytoD_6s_2021-2', 'DMSO_6s_2024-2', 'DMSO_6s_2024-3', 'elastase_3s_2021', 'Gd_6s_2021', 'GluC_3s_2021', 'MBCD_6s_2021-3']


In [7]:
print(f"Training samples: {len(train_dataset_list)}")
print(f"Validation sampples: {len(val_dataset_list)}")

Training samples: 2364
Validation sampples: 591


In [8]:
train_transforms = t.Compose(
    [
        t.LoadImaged(["image", "label"]),
        t.EnsureChannelFirstd(["image", "label"]),
        t.ResizeWithPadOrCropd(["image", "label"], spatial_size=(348, 348)),
        t.Resized(["image", "label"],
                  spatial_size=(192, 192),
                  mode=["bilinear", "nearest"],
                  ),
        t.ScaleIntensityRanged(["image"], 0, 255, 0, 1),
        t.RandFlipd(["image", "label"]),
        t.RandGaussianSmoothd(["image"]),
        t.RandShiftIntensityd(["image"], prob=0.2, offsets=0.3),
        t.Rand2DElasticd(
            ["image", "label"],
            spacing=(20, 20),
            magnitude_range=[1, 2],
            mode=["bilinear", "nearest"]),
        t.RandAffined(
            ["image", "label"],
             prob=0.2,
            rotate_range=180,
            translate_range=20,
            padding_mode="zeros",
            cache_grid=True,
            mode=["bilinear", "nearest"],
        ),
        t.ToTensord(["image", "label"]),
    ]
)

val_transforms = t.Compose(
    [
        t.LoadImaged(["image", "label"]),
        t.EnsureChannelFirstd(["image", "label"]),
        t.ResizeWithPadOrCropd(["image", "label"], spatial_size=(348, 348)),
        t.Resized(["image", "label"],
                  spatial_size=(192, 192),
                  mode=["bilinear", "nearest"],
                  ),
        t.ScaleIntensityRanged(["image"], 0, 255, 0, 1),
        t.ToTensord(["image", "label"]),
    ]
)

C:\Users\siniz\AppData\Roaming\Python\Python310\site-packages\monai\transforms\spatial\array.py:2426: UserWarning: cache_grid=True is not compatible with the dynamic spatial_size, please specify 'spatial_size'.
  warnings.warn(


In [9]:
train_ds = CacheDataset(train_dataset_list,
                        # cache_rate=0.1,
                        transform=train_transforms)
val_ds = CacheDataset(val_dataset_list,
                      transform=val_transforms)

Loading dataset: 100%|██████████| 591/591 [00:02<00:00, 221.03it/s]


In [10]:
from notebooks.lit_unet import LitUNet

In [11]:
train_loader = DataLoader(train_ds,
                          batch_size=16,
                          num_workers=0,
                          shuffle=True)
val_loader = DataLoader(val_ds,
                        batch_size=16,
                        num_workers=0)

In [12]:
# create a model with specified hparams and trainer
lit_model = LitUNet(n_classes=2,
                    num_input_channel=1,
                    check_val=10,
                    max_epochs=200,)

In [ ]:
lr_logger = LearningRateMonitor()
checkpoint_callback = ModelCheckpoint(
        filename='checkpoint-' +
        f"cell_seg_unet" + '-{epoch:02d}-{val_loss:.2f}',
        monitor='val_loss',
        mode='min')
logger = TensorBoardLogger("tb_logs", name="cell_unet_model_small")

trainer = L.Trainer(
    logger=logger,
    log_every_n_steps=2,
    callbacks=[checkpoint_callback, lr_logger],
    precision="16-mixed",
    max_epochs=lit_model.hparams.max_epochs,
    gradient_clip_val=10,
    check_val_every_n_epoch=lit_model.hparams.check_val
)
# start training the model
trainer.fit(lit_model, train_loader, val_loader)

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 4060 Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name  | Type            | Params | Mode 
--------------------------------------------------
0 | model | UNet            | 6.4 M  | train
1 | loss  | HausdorffDTLoss | 0      | train
--------------------------------------------------
6.4 M     Trainable params
0         Non-trainable params
6.4 M     Total params
25.619    Total estimated model params size (MB)
112       Modules in train mode
0         Modules in ev

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

c:\Program Files\Python310\lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:424: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=21` in the `DataLoader` to improve performance.


current epoch: 0 current mean dice: 67.8036
best mean dice: 67.8036 at epoch: 0


c:\Program Files\Python310\lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:424: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=21` in the `DataLoader` to improve performance.


...

current epoch: 109 current mean dice: 3.9701
best mean dice: 3.9701 at epoch: 109


Validation: |          | 0/? [00:00<?, ?it/s]

current epoch: 119 current mean dice: 3.9654
best mean dice: 3.9654 at epoch: 119


Validation: |          | 0/? [00:00<?, ?it/s]

current epoch: 129 current mean dice: 5.3910
best mean dice: 3.9654 at epoch: 119


In [ ]:
lit_model.plot_losses()

In [ ]:
checkpoint_callback.best_model_path

In [ ]:
best_model = LitUNet.load_from_checkpoint(checkpoint_callback.best_model_path)

In [ ]:
val_ds[0]['image'].unsqueeze(0).shape

In [ ]:
def enable_running_stats(model, device):
    """Temporarily enable track_running_stats and initialize running statistics."""
    for module in model.modules():
        if isinstance(module, torch.nn.InstanceNorm2d):  # Adjust for InstanceNorm1d/3d if needed
            num_features = module.num_features
            module.running_mean = torch.zeros(num_features, device=device)
            module.running_var = torch.ones(num_features, device=device)
            module.track_running_stats = True